# Skill 39: Key-Value (KV) Caching

**Phase 3 - LLMs & Generative AI Applications (Skills 31-45)**

## What this skill is about
Reuse previously computed keys and values so autoregressive decoding does not recompute the whole prefix.

## Objectives
- Explain why past K/V tensors remain valid when a new token is appended.
- Implement append-only K/V cache logic.
- Compare projected token positions with and without caching.

Run the cells top to bottom. Requires `torch`, `numpy` and `matplotlib` (`pip install torch numpy matplotlib`).

## Theory and core formulas
$$o_t=\mathrm{softmax}\!\Big(\frac{q_tK_{1:t}^{\top}}{\sqrt d}\Big)V_{1:t}$$
Projection work: $T(T+1)/2$ without cache versus $T$ with cache. Cache memory: $2\,L\,B\,T\,E\,b$ bytes.

## Implementation
Reference code from the lab manual (unchanged).

In [ ]:
import torch
import torch.nn as nn
import math

class CachedSelfAttention(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.q = nn.Linear(dim, dim)
        self.k = nn.Linear(dim, dim)
        self.v = nn.Linear(dim, dim)
        self.out = nn.Linear(dim, dim)

    def step(self, x_new, cache=None):
        # x_new: (B, 1, D)
        q = self.q(x_new)
        k_new, v_new = self.k(x_new), self.v(x_new)
        if cache is None:
            k_all, v_all = k_new, v_new
        else:
            k_all = torch.cat([cache[0], k_new], dim=1)
            v_all = torch.cat([cache[1], v_new], dim=1)
        scores = q @ k_all.transpose(-2, -1) / math.sqrt(q.size(-1))
        weights = scores.softmax(dim=-1)
        y = self.out(weights @ v_all)
        return y, (k_all, v_all)

torch.manual_seed(0)
layer = CachedSelfAttention(16)
tokens = torch.randn(1, 8, 16)
cache, outputs = None, []
for i in range(tokens.size(1)):
    y, cache = layer.step(tokens[:, i:i+1], cache)
    outputs.append(y)
print(torch.cat(outputs, dim=1).shape)  # (1,8,16)
T = 8
print("naive projected positions:", T * (T + 1) // 2)
print("cached projected positions:", T)


## What to check
**Expected result:** One output vector per token; naive projections 36 versus cached 8 for T=8.

## Pitfalls
Single layer/head demo. Real decoders cache per layer and need the absolute position. `torch.cat` copies the cache each step. Clear the cache for new sequences.